# 06 - Lakeflow Designer: Preparación Visual de Datos

## Objetivo
Crear una transformación **sin código** con el canvas visual de **Lakeflow Designer**.

## Prerequisitos
- Catálogo **BNS** con datos (después de `00_setup` y Lab 04)
- Permiso `CAN USE` en compute Serverless

## Duración: ~20 minutos


In [ ]:
%run ../00\ -\ Setup/00_variables


In [ ]:
# Datos de entrada para Designer
display(spark.sql(f"SELECT * FROM {catalog_name}.silver.transacciones LIMIT 5"))


## Paso 1 — Crear Visual Data Prep

1. En el menú lateral, clic en **New**
2. Seleccione **Visual data prep** (Lakeflow Designer)
3. Nombre: `BNCR Resumen Canales - <usuario>`

## Paso 2 — Conectar fuente de datos

1. En el canvas, agregue un nodo **Read**
2. Seleccione como fuente:
   - **Catálogo:** `BNS`
   - **Esquema:** `silver`
   - **Tabla:** `transacciones`
3. Clic en **Preview** para ver los datos

## Paso 3 — Transformar (drag & drop)

Agregue operadores en este orden:

| # | Operador | Configuración |
|---|----------|---------------|
| 1 | **Filter** | `monto > 0` y `moneda = 'CRC'` |
| 2 | **Group by** | Agrupar por `canal`, `fecha_transaccion` |
| 3 | **Aggregate** | `COUNT(*)` → `total_txn`, `SUM(monto)` → `monto_total` |
| 4 | **Sort** | Por `monto_total` descendente |

> También puede describir la transformación en **lenguaje natural** en el panel de Genie y Designer generará los pasos.

## Paso 4 — Escribir resultado

1. Agregue nodo **Write**
2. Destino Unity Catalog:
   - Catálogo: `BNS`
   - Esquema: `gold`
   - Tabla: `designer_resumen_canales`
3. Modo: **Overwrite** o **Append**
4. **Preview** final → confirmar

## Paso 5 — Guardar y ejecutar

1. **Save** — se guarda como archivo `.designer.ipynb`
2. Clic **Run** para ejecutar la transformación
3. Opcional: **Schedule** para programar como Job


In [ ]:
# Verificar tabla creada por Designer
try:
    display(spark.sql(f"""
      SELECT canal, fecha_transaccion, total_txn, monto_total
      FROM {catalog_name}.gold.designer_resumen_canales
      ORDER BY monto_total DESC
      LIMIT 10
    """))
    print("Lakeflow Designer OK")
except Exception as e:
    print("Complete los pasos del Designer primero.")
    print(f"Detalle: {e}")


## Paso 6 — Agregar a un Job (opcional)

1. **Jobs & Pipelines → Create → Job**
2. Agregar tarea tipo **Notebook**
3. Seleccionar su archivo `.designer.ipynb`
4. Programar o ejecutar manualmente

## Comparación rápida

| Enfoque | Cuándo usar |
|---------|-------------|
| **SQL manual (Lab 04)** | Control total, producción |
| **Genie Code (Lab 05)** | Crear/editar pipelines con IA |
| **Lakeflow Designer (Lab 06)** | Usuarios de negocio, exploración visual |

## Recursos
- [Lakeflow Designer](https://docs.databricks.com/aws/en/designer/)
- [Crear visual data prep](https://docs.databricks.com/aws/en/designer/build-transformation)
